# 원본 PCB 이미지 한 장의 정상/불량 분류

입력은 **원본 640×640 검사 이미지 한 장**입니다. 추론할 때 정상 기준 사진이나 불량 좌표를 받지 않습니다.

- 학습 라벨: 정상 기준 원본 `_temp.jpg`는 0, 불량 검사 원본 `_test.jpg`는 1.
- 원본을 Crop하거나 128×128로 축소하지 않고 640×640 전체를 읽습니다.
- 기존 이진 패치 CNN의 합성곱 가중치를 활용하고, 원본 전체 특징의 최대값/평균값을 여러 해상도에서 모아 새 분류기를 학습합니다.
- 특징 고정 전이학습과 마지막 합성곱 층 미세조정을 동일 검증 그룹에서 비교합니다.
- 같은 원본 쌍은 항상 같은 split이며 `group00041` 전체를 validation으로 유지합니다. 공식 test는 학습/선택에 사용하지 않습니다.
- 목표 지표는 validation Macro F1이며 0.90 이상을 지향하되 보장하지 않습니다.

정상 원본이 모두 기준 영상이라는 데이터 출처 제한이 있습니다. 한 장의 정상 검사 사진을 현장에서 사용하는 성능은 별도 검증이 필요합니다. 모든 입력·증강 정책은 두 라벨에 동일하게 적용합니다.

In [1]:
# 기존 Python과 Metal 회피 설정을 사용합니다.
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display, Image as DisplayImage, Markdown
from src.runtime import configure_runtime

ROOT = Path.cwd()
print('Python:', sys.version)
print('TensorFlow:', tf.__version__)
print(configure_runtime())

Python: 3.11.9 (main, Aug 14 2024, 04:17:21) [Clang 18.1.8 ]
TensorFlow: 2.18.1
{'optimizer_options': {'disable_model_pruning': False, 'disable_meta_optimizer': True}, 'gpu_devices': ["PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')"]}


In [2]:
# 기존 원본 분할을 유지하고 정상/불량 원본 파일의 manifest를 작성합니다.
from src.full_image_data import prepare_full_image_data

full_data_summary = prepare_full_image_data(ROOT)
full_frame = pd.read_csv(ROOT / 'data/split/full_image_v1/images.csv')
display(pd.crosstab(full_frame.split, full_frame.label))
print('입력 해상도:', full_data_summary['native_resolution'])
print('원본 Crop 여부:', full_data_summary['cropped'])

{
  "counts": [
    {
      "split": "train",
      "label": "defect",
      "images": 800
    },
    {
      "split": "train",
      "label": "normal",
      "images": 800
    },
    {
      "split": "val",
      "label": "defect",
      "images": 200
    },
    {
      "split": "val",
      "label": "normal",
      "images": 200
    }
  ],
  "rows": 2000,
  "duplicate_pixel_rows_within_train": 2,
  "train_sources": 800,
  "validation_sources": 200,
  "cross_split_pixel_overlap": 0,
  "cross_split_group_overlap": 0,
  "test_images_read": 0,
  "native_resolution": [
    640,
    640
  ],
  "cropped": false,
  "manifest_sha256": "4ce69ca689f9825e86d545edb56415f0b34f3716564579376d257c124ceaa753"
}


label,defect,normal
split,,
train,800,800
val,200,200


입력 해상도: [640, 640]
원본 Crop 여부: False


## 1차 — 원본 전체 특징으로 분류기 학습

기존 패치 CNN에서 합성곱 가중치만 복제합니다. Flatten 이후의 패치 분류기는 가져오지 않습니다. 원본 전체를 넣고 각 블록의 GlobalMax/GlobalAverage 특징을 모아 새 분류기를 학습합니다.

- train: 원본 및 전체 90도 회전 두 view. validation: 원본만.
- 원본 640×640을 batch 2로 읽고 작은 특징 벡터만 캐시합니다.
- 특징의 평균/분산 정규화는 train 특징으로만 계산합니다.
- 새 분류기는 CPU에서 학습해 메모리와 GPU 호출 비용을 줄입니다.
- 최종 저장 모델은 합성곱 특징 추출기와 분류기를 합친 단일 640×640 입력 모델입니다.

In [3]:
# 원본 이미지 1,600장으로부터 특징을 추출하고 새 분류기를 학습합니다.
import importlib
import src.full_image_data
importlib.reload(src.full_image_data)
from src.full_image_train import train_full_frozen

frozen_result = train_full_frozen(ROOT)
display(frozen_result)

2026-11-18 16:52:34.894289: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1 Pro
2026-11-18 16:52:34.894328: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-11-18 16:52:34.894339: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
I0000 00:00:1794988354.894511 17225372 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1794988354.894681 17225372 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


train 원본 특징 추출: 200/1600장
train 원본 특징 추출: 400/1600장
train 원본 특징 추출: 600/1600장
train 원본 특징 추출: 800/1600장
train 원본 특징 추출: 1000/1600장
train 원본 특징 추출: 1200/1600장
train 원본 특징 추출: 1400/1600장
train 원본 특징 추출: 1600/1600장


2026-11-18 16:53:15.557038: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


val 원본 특징 추출: 200/400장
val 원본 특징 추출: 400/400장
원본 전체 특징: train=(3200, 224), validation=(400, 224)


2026-11-18 16:53:20.077651: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


epoch=01 val_acc=0.9100 macro_f1=0.9093 defect_recall=0.9950 normal_fpr=0.1750 elapsed=0s
epoch=02 val_acc=0.9350 macro_f1=0.9348 defect_recall=0.9900 normal_fpr=0.1200 elapsed=1s
epoch=03 val_acc=0.9475 macro_f1=0.9474 defect_recall=0.9950 normal_fpr=0.1000 elapsed=1s
epoch=04 val_acc=0.9525 macro_f1=0.9524 defect_recall=0.9950 normal_fpr=0.0900 elapsed=1s
epoch=05 val_acc=0.9500 macro_f1=0.9499 defect_recall=0.9950 normal_fpr=0.0950 elapsed=1s
epoch=06 val_acc=0.9525 macro_f1=0.9524 defect_recall=0.9900 normal_fpr=0.0850 elapsed=1s
epoch=07 val_acc=0.9500 macro_f1=0.9499 defect_recall=0.9900 normal_fpr=0.0900 elapsed=1s
epoch=08 val_acc=0.9450 macro_f1=0.9449 defect_recall=0.9950 normal_fpr=0.1050 elapsed=1s
epoch=09 val_acc=0.9475 macro_f1=0.9474 defect_recall=0.9950 normal_fpr=0.1000 elapsed=1s
epoch=10 val_acc=0.9500 macro_f1=0.9499 defect_recall=0.9950 normal_fpr=0.0950 elapsed=1s
epoch=11 val_acc=0.9500 macro_f1=0.9499 defect_recall=0.9950 normal_fpr=0.0950 elapsed=1s
epoch=12 v

{'run_name': 'full_frozen_001',
 'metrics': {'samples': 400,
  'accuracy': 0.9525,
  'balanced_accuracy': 0.9525,
  'defect_precision': 0.9209302325581395,
  'defect_recall': 0.99,
  'defect_f1': 0.9542168674698795,
  'macro_f1': 0.9524331090596151,
  'normal_false_positive_rate': 0.085,
  'defect_false_negative_rate': 0.01,
  'roc_auc': 0.978625,
  'average_precision': 0.9596309738973257,
  'true_normal': 183,
  'false_defect': 17,
  'missed_defect': 2,
  'true_defect': 198,
  'threshold': 0.5,
  'split': 'validation',
  'unit': 'full_640_image',
  'official_test_evaluated': False},
 'epochs_completed': 12,
 'best_epoch': 6,
 'elapsed_seconds': 52.493929582997225,
 'full_model_parameters': 86929,
 'trainable_parameters': 14465,
 'cached_vs_end_to_end_max_score_delta': 2.1483612056005796e-07,
 'model_path': '/Users/yeongjunna/DNN_project/models/full_frozen_001/best.keras',
 'status': 'completed'}

## 2차 — 원본 640×640으로 마지막 합성곱 층 미세조정

1차 모델에서 시작해 마지막 Conv2D 두 층과 분류기만 작은 학습률로 조정합니다. BN 통계는 고정하고, 원본 전체 90도 회전/반전을 train에만 적용합니다. 입력은 계속 단일 640×640 원본이며 별도 패치 파일이나 좌표를 사용하지 않습니다.

- batch 2, Adam 0.00001, 최대 6 epoch, EarlyStopping patience 3.
- 1차 checkpoint를 epoch 0 후보로 보존합니다. 개선되지 않으면 초기 모델을 유지합니다.
- 같은 validation Macro F1로 비교하며 공식 test는 사용하지 않습니다.

In [4]:
# 원본 전체를 네트워크에 넣고 선택한 합성곱 층까지 학습합니다.
from src.full_image_train import train_full_finetuned

finetuned_result = train_full_finetuned(ROOT)
display(finetuned_result)

원본 640×640 미세조정 시작: train=1600, val=400, Conv=['full_conv2d_4', 'full_conv2d_5']
epoch=01 val_acc=0.9500 macro_f1=0.9499 defect_recall=0.9950 normal_fpr=0.0950 elapsed=41s


2026-11-18 16:56:06.268730: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


epoch=02 val_acc=0.9500 macro_f1=0.9499 defect_recall=0.9950 normal_fpr=0.0950 elapsed=79s
epoch=03 val_acc=0.9500 macro_f1=0.9499 defect_recall=0.9950 normal_fpr=0.0950 elapsed=116s
epoch=04 val_acc=0.9550 macro_f1=0.9549 defect_recall=0.9950 normal_fpr=0.0850 elapsed=152s
epoch=05 val_acc=0.9500 macro_f1=0.9499 defect_recall=0.9950 normal_fpr=0.0950 elapsed=189s


2026-11-18 16:58:34.120627: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


epoch=06 val_acc=0.9425 macro_f1=0.9423 defect_recall=0.9950 normal_fpr=0.1100 elapsed=226s
{
  "run_name": "full_finetuned_001",
  "metrics": {
    "samples": 400,
    "accuracy": 0.955,
    "balanced_accuracy": 0.9550000000000001,
    "defect_precision": 0.9212962962962963,
    "defect_recall": 0.995,
    "defect_f1": 0.9567307692307693,
    "macro_f1": 0.9549278846153846,
    "normal_false_positive_rate": 0.085,
    "defect_false_negative_rate": 0.005,
    "roc_auc": 0.979325,
    "average_precision": 0.9611360193770446,
    "true_normal": 183,
    "false_defect": 17,
    "missed_defect": 1,
    "true_defect": 199,
    "threshold": 0.5,
    "split": "validation",
    "unit": "full_640_image",
    "official_test_evaluated": false
  },
  "epochs_completed": 6,
  "best_epoch": 4,
  "baseline_macro_f1": 0.9524331090596151,
  "trainable_convolutions": [
    "full_conv2d_4",
    "full_conv2d_5"
  ],
  "elapsed_seconds": 231.35272425005678,
  "model_path": "/Users/yeongjunna/DNN_project/mo

{'run_name': 'full_finetuned_001',
 'metrics': {'samples': 400,
  'accuracy': 0.955,
  'balanced_accuracy': 0.9550000000000001,
  'defect_precision': 0.9212962962962963,
  'defect_recall': 0.995,
  'defect_f1': 0.9567307692307693,
  'macro_f1': 0.9549278846153846,
  'normal_false_positive_rate': 0.085,
  'defect_false_negative_rate': 0.005,
  'roc_auc': 0.979325,
  'average_precision': 0.9611360193770446,
  'true_normal': 183,
  'false_defect': 17,
  'missed_defect': 1,
  'true_defect': 199,
  'threshold': 0.5,
  'split': 'validation',
  'unit': 'full_640_image',
  'official_test_evaluated': False},
 'epochs_completed': 6,
 'best_epoch': 4,
 'baseline_macro_f1': 0.9524331090596151,
 'trainable_convolutions': ['full_conv2d_4', 'full_conv2d_5'],
 'elapsed_seconds': 231.35272425005678,
 'model_path': '/Users/yeongjunna/DNN_project/models/full_finetuned_001/best.keras',
 'status': 'completed'}